In [ ]:
from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/paper.json').is_file())
os.chdir(root)
n_bootstrap = 5000
block_lengths = [5, 10, 20]
confidence_level = 0.95
random_seed = 20261027
excluded_month = '2025-04'
output = Path('Results/Sensitivity')
folder = Path('FINDER/Official')
scores = pd.read_csv(folder / 'daily_scores_all_markets.csv')
metrics = pd.read_csv(folder / 'market_agent_metrics.csv')

In [ ]:
markets = {'^GSPC': 'SP500', 'FTSE100': 'FTSE', 'GDAXI': 'DAX', 'CACT': 'CAC', 'STOXX': 'STOXX', 'CBU0.L': 'CBU0'}
models = ['decision_supervisor', 'quantitative_expert', 'chart_expert', 'news_reader', 'news_expert', 'lstm_ind', 'lstm_news']
metric_names = ['roc_auc', 'ap']
frames = []
for market, code in markets.items():
    frame = scores.loc[scores.market.eq(market), ['target_date', 'actual_label', *models[:5]]].copy()
    assert len(frame) and not frame.target_date.duplicated().any(), market
    for variant, model in [('IND', 'lstm_ind'), ('NEWS', 'lstm_news')]:
        benchmark = pd.read_csv(Path('Benchmark/Official') / f'{code}_{variant}' / 'test_predictions.csv')
        assert set(frame.target_date) == set(benchmark.target_date), (market, model)
        frame = frame.merge(benchmark[['target_date', 'actual', 'probability']], on='target_date', validate='one_to_one')
        assert frame.actual_label.eq(frame.actual).all(), (market, model)
        frame = frame.drop(columns='actual').rename(columns={'probability': model})
    frame['target_date'] = pd.to_datetime(frame.target_date)
    frame = frame.sort_values('target_date').set_index('target_date')
    assert frame.index.is_unique and frame.actual_label.isin([0, 1]).all(), market
    assert frame.actual_label.nunique() == 2 and np.isfinite(frame.to_numpy(dtype=float)).all(), market
    frames.append(frame)

dates = pd.DatetimeIndex(sorted(set().union(*(frame.index for frame in frames))))
panel = np.stack([frame.reindex(dates)[['actual_label', *models]].to_numpy(dtype=float) for frame in frames])
sample_sizes = np.array([len(frame) for frame in frames])
event_counts = np.array([frame.actual_label.sum() for frame in frames], dtype=int)
point_metrics = np.empty((len(markets) + 1, len(models), len(metric_names)))
benchmark_metrics = pd.read_csv('Benchmark/Official/metrics.csv')
for m, (market, code) in enumerate(markets.items()):
    actual = frames[m].actual_label.to_numpy()
    for j, model in enumerate(models):
        probability = frames[m][model].to_numpy()
        point_metrics[m, j] = [roc_auc_score(actual, probability), average_precision_score(actual, probability)]
        if model.startswith('lstm_'):
            variant = 'IND' if model == 'lstm_ind' else 'NEWS'
            expected = benchmark_metrics.loc[benchmark_metrics.market.eq(code) & benchmark_metrics.model.eq(variant)]
        else:
            expected = metrics.loc[metrics.market.eq(market) & metrics.agent.eq(model)]
        assert len(expected) == 1 and np.allclose(point_metrics[m, j], expected[metric_names].iloc[0]), (market, model)
point_metrics[-1] = point_metrics[:-1].mean(axis=0)
display(pd.DataFrame({'market': list(markets), 'n': sample_sizes, 'events': event_counts, 'event_rate': event_counts / sample_sizes}))


In [ ]:
def weighted_metrics(sample_weights, sample_panel):
    result = np.full((len(sample_weights), len(markets) + 1, len(models), 2), np.nan)
    for m in range(len(markets)):
        valid = np.isfinite(sample_panel[m, :, 0])
        actual = sample_panel[m, valid, 0]
        weights = sample_weights[:, valid]
        positive_total = weights @ actual
        negative_total = weights.sum(axis=1) - positive_total
        valid_draw = (positive_total > 0) & (negative_total > 0)
        for j in range(len(models)):
            probability = sample_panel[m, valid, j + 1]
            order = np.argsort(-probability, kind='stable')
            starts = np.r_[0, np.flatnonzero(np.diff(probability[order])) + 1]
            counts = np.add.reduceat(weights[:, order], starts, axis=1)
            positives = np.add.reduceat(weights[:, order] * actual[order], starts, axis=1)
            negatives = counts - positives
            cumulative_positives = positives.cumsum(axis=1)
            cumulative_counts = counts.cumsum(axis=1)
            with np.errstate(divide='ignore', invalid='ignore'):
                auc = (positives * (negative_total[:, None] - negatives.cumsum(axis=1) + 0.5 * negatives)).sum(axis=1) / (positive_total * negative_total)
                precision = np.divide(cumulative_positives, cumulative_counts, out=np.zeros_like(cumulative_positives), where=cumulative_counts > 0)
                ap = (positives * precision).sum(axis=1) / positive_total
            result[valid_draw, m, j, 0] = auc[valid_draw]
            result[valid_draw, m, j, 1] = ap[valid_draw]
    result[:, -1] = result[:, :-1].mean(axis=1)
    return result

# Frequency weights reproduce duplicated observations, including tied score groups.
assert np.allclose(weighted_metrics(np.ones((1, len(dates)), dtype=int), panel)[0], point_metrics, atol=1e-12)
check_rng = np.random.default_rng(28)
for check in range(12):
    indices = check_rng.integers(len(dates), size=len(dates))
    weights = np.bincount(indices, minlength=len(dates))[None, :]
    calculated = weighted_metrics(weights, panel)[0]
    for m in range(len(markets)):
        sample = panel[m, indices]
        sample = sample[np.isfinite(sample[:, 0])]
        for j in range(len(models)):
            expected = [roc_auc_score(sample[:, 0], sample[:, j + 1]), average_precision_score(sample[:, 0], sample[:, j + 1])]
            assert np.allclose(calculated[m, j], expected, atol=1e-12)
constant_panel = panel.copy()
constant_panel[:, :, 1:] = 0.5
constant_metrics = weighted_metrics(np.ones((1, len(dates)), dtype=int), constant_panel)[0]
assert np.allclose(constant_metrics[:-1, :, 0], 0.5)
assert np.allclose(constant_metrics[:-1, :, 1], (event_counts / sample_sizes)[:, None])
zero_panel = panel.copy()
zero_panel[:, :, 0] = np.where(np.isfinite(zero_panel[:, :, 0]), 0, np.nan)
assert np.isnan(weighted_metrics(np.ones((1, len(dates)), dtype=int), zero_panel)).all()
print('Weighted rankings match scikit-learn on original data and 12 shared-date resamples; ties and single-class draws checked.')

In [ ]:
month_event_totals = scores.assign(month=scores.target_date.str[:7]).groupby('month').actual_label.sum()
additional_month = month_event_totals.drop(excluded_month).idxmax()
scenarios = [('full_l10', 10, None), ('exclude_2025_04_l10', 10, excluded_month), ('exclude_' + additional_month.replace('-', '_') + '_l10', 10, additional_month), ('full_l5', 5, None), ('full_l20', 20, None)]
metric_rows, difference_rows, sample_rows = [], [], []
quantiles = [(1 - confidence_level) / 2, (1 + confidence_level) / 2]

for scenario, block_length, removed_month in scenarios:
    scenario_panel = panel.copy()
    removed = dates.to_period('M').astype(str) == removed_month if removed_month else np.zeros(len(dates), dtype=bool)
    scenario_panel[:, removed, :] = np.nan
    sizes = np.isfinite(scenario_panel[:, :, 0]).sum(axis=1)
    events = np.nansum(scenario_panel[:, :, 0], axis=1).astype(int)
    assert ((events > 0) & (events < sizes)).all()
    points = weighted_metrics(np.ones((1, len(dates)), dtype=int), scenario_panel)[0]
    rng = np.random.default_rng(random_seed)
    weights = np.zeros((n_bootstrap, len(dates)), dtype=np.int32)
    for b in range(n_bootstrap):
        indices = rng.integers(len(dates), size=len(dates))
        restart = rng.random(len(dates)) < 1 / block_length
        for t in range(1, len(dates)):
            if not restart[t]:
                indices[t] = (indices[t - 1] + 1) % len(dates)
        weights[b] = np.bincount(indices, minlength=len(dates))
    bootstrap = weighted_metrics(weights, scenario_panel)
    differences = points[:, :1] - points[:, 1:]
    bootstrap_differences = bootstrap[:, :, :1] - bootstrap[:, :, 1:]
    settings = {'scenario': scenario, 'block_length': block_length, 'excluded_month': removed_month or '', 'n_bootstrap': n_bootstrap, 'confidence_level': confidence_level, 'random_seed': random_seed, 'block_unit': 'original_union_date_observations'}
    for m, market in enumerate([*markets, 'equal_weight_mean']):
        aggregate = m == len(markets)
        context = {'market': market, 'n': None if aggregate else int(sizes[m]), 'events': None if aggregate else int(events[m]), 'event_rate': float((events / sizes).mean() if aggregate else events[m] / sizes[m]), **settings}
        sample_rows.append(context)
        for j, model in enumerate(models):
            for k, metric in enumerate(metric_names):
                values = bootstrap[:, m, j, k]
                valid = values[np.isfinite(values)]
                bounds = np.quantile(valid, quantiles) if len(valid) >= 2 else [np.nan, np.nan]
                metric_rows.append({**context, 'model': model, 'metric': metric, 'estimate': points[m, j, k], 'ci_lower': bounds[0], 'ci_upper': bounds[1], 'n_valid': len(valid), 'n_invalid': n_bootstrap - len(valid)})
        for j, comparator in enumerate(models[1:]):
            for k, metric in enumerate(metric_names):
                values = bootstrap_differences[:, m, j, k]
                valid = values[np.isfinite(values)]
                bounds = np.quantile(valid, quantiles) if len(valid) >= 2 else [np.nan, np.nan]
                difference_rows.append({**context, 'model': models[0], 'comparator': comparator, 'metric': metric, 'estimate': differences[m, j, k], 'ci_lower': bounds[0], 'ci_upper': bounds[1], 'n_valid': len(valid), 'n_invalid': n_bootstrap - len(valid)})
    print(f'{scenario}: {n_bootstrap} draws; equal-weight valid = {np.isfinite(bootstrap[:, -1, 0, 0]).sum()}')

sensitivity_metrics = pd.DataFrame(metric_rows)
sensitivity_pairs = pd.DataFrame(difference_rows)
sensitivity_samples = pd.DataFrame(sample_rows)
for current, name, key in [(sensitivity_metrics, 'metrics_ci.csv', ['market', 'model', 'metric']), (sensitivity_pairs, 'paired_differences_ci.csv', ['market', 'comparator', 'metric'])]:
    saved = pd.read_csv(Path('FINDER/Uncertainty') / name).set_index(key).sort_index()
    baseline = current[current.scenario.eq('full_l10')].set_index(key).sort_index()
    assert baseline.index.equals(saved.index)
    assert np.allclose(baseline[['estimate', 'ci_lower', 'ci_upper', 'n_valid', 'n_invalid']], saved[['estimate', 'ci_lower', 'ci_upper', 'n_valid', 'n_invalid']], atol=1e-12)
print('Original L=10 estimates, confidence limits and valid-draw counts reproduced.')
display(sensitivity_pairs[sensitivity_pairs.market.eq('equal_weight_mean') & sensitivity_pairs.comparator.isin(['lstm_ind', 'lstm_news', 'quantitative_expert'])][['scenario', 'comparator', 'metric', 'estimate', 'ci_lower', 'ci_upper', 'n_valid']].round(4))

In [ ]:
monthly_rows, monthly_pair_rows, concentration_rows = [], [], []
for month in dates.to_period('M').unique().astype(str):
    removed = dates.to_period('M').astype(str) == month
    scenario_panel = panel.copy()
    scenario_panel[:, removed, :] = np.nan
    points = weighted_metrics(np.ones((1, len(dates)), dtype=int), scenario_panel)[0]
    sizes = np.isfinite(scenario_panel[:, :, 0]).sum(axis=1)
    events = np.nansum(scenario_panel[:, :, 0], axis=1).astype(int)
    for m, market in enumerate([*markets, 'equal_weight_mean']):
        aggregate = m == len(markets)
        context = {'excluded_month': month, 'market': market, 'n': None if aggregate else int(sizes[m]), 'events': None if aggregate else int(events[m]), 'event_rate': float((events / sizes).mean() if aggregate else events[m] / sizes[m])}
        for j, model in enumerate(models):
            for k, metric in enumerate(metric_names):
                monthly_rows.append({**context, 'model': model, 'metric': metric, 'estimate': points[m, j, k]})
        for j, comparator in enumerate(models[1:], 1):
            for k, metric in enumerate(metric_names):
                monthly_pair_rows.append({**context, 'comparator': comparator, 'metric': metric, 'estimate': points[m, 0, k] - points[m, j, k]})
        if not aggregate:
            removed_n = int(np.isfinite(panel[m, removed, 0]).sum())
            removed_events = int(np.nansum(panel[m, removed, 0]))
            concentration_rows.append({'month': month, 'market': market, 'n': removed_n, 'events': removed_events, 'share_of_market_events': removed_events / event_counts[m]})
monthly_metrics = pd.DataFrame(monthly_rows)
monthly_pairs = pd.DataFrame(monthly_pair_rows)
monthly_events = pd.DataFrame(concentration_rows)
assert monthly_metrics.estimate.notna().all() and monthly_pairs.estimate.notna().all()
print(f'Descriptive leave-one-calendar-month-out check: {len(monthly_events.month.unique())} months; no month selected using model performance.')
display(monthly_events.groupby('month')[['n', 'events']].sum())
display(monthly_pairs[monthly_pairs.market.eq('equal_weight_mean') & monthly_pairs.comparator.isin(['lstm_ind', 'lstm_news'])].pivot(index='excluded_month', columns=['comparator', 'metric'], values='estimate').round(4))

In [ ]:
output.mkdir(parents=True, exist_ok=True)
for table, name in [(sensitivity_metrics, 'metrics_ci.csv'), (sensitivity_pairs, 'paired_differences_ci.csv'), (sensitivity_samples, 'samples.csv'), (monthly_metrics, 'monthly_metrics.csv'), (monthly_pairs, 'monthly_paired_differences.csv'), (monthly_events, 'monthly_event_counts.csv')]:
    table.to_csv(output / name, index=False)
inputs = [folder / 'daily_scores_all_markets.csv', folder / 'market_agent_metrics.csv', Path('Benchmark/Official/metrics.csv'), *sorted(Path('Benchmark/Official').glob('*/test_predictions.csv'))]
record = {'n_bootstrap': n_bootstrap, 'block_lengths': block_lengths, 'random_seed': random_seed, 'primary_excluded_month': excluded_month, 'additional_excluded_month': additional_month, 'additional_month_selection': 'largest sum of market event counts outside April 2025; independent of model scores', 'resampling_grid': 'original union of target dates; excluded April dates remain masked', 'monthly_check': 'each calendar month omitted once; descriptive point estimates only', 'aggregation': 'equal weight 1/6; invalid if any market lacks either class', 'predictions_and_labels': 'saved scores and event labels', 'intervals': 'pointwise percentile; conditional on saved predictions', 'source_sha256': {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in inputs}}
(output / 'method.json').write_text(json.dumps(record, indent=2) + '\n')
print(f'Saved {len(sensitivity_metrics)} metric intervals, {len(sensitivity_pairs)} paired intervals and monthly checks to {output}.')